In [20]:
from typing import Dict, Optional, List
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# FUNÇÕES DE CARREGAMENTO E PRÉ-PROCESSAMENTO
# ============================================================

def load_data(file_paths: Dict[str, str]) -> Dict[str, pd.DataFrame]:
    """Carrega múltiplos arquivos CSV"""
    datasets: Dict[str, pd.DataFrame] = {}
    for name, path in file_paths.items():
        datasets[name] = pd.read_csv(path, header="infer")
    return datasets

def convert_last_n_columns_to_numeric(df: pd.DataFrame, n: int = 3, errors: str = 'coerce') -> pd.DataFrame:
    """Converte as últimas n colunas para tipo numérico"""
    df.iloc[:, -n:] = df.iloc[:, -n:].apply(pd.to_numeric, errors=errors)
    return df

def clean_data_with_dropna(df: pd.DataFrame, name: str = "Dataset") -> pd.DataFrame:
    """Remove linhas com NaN e valores inválidos, mostra a porcentagem removida"""
    rows_before: int = len(df)
    
    # Substituir valores inválidos por NaN
    invalid_values: List[str] = ["...", "..", "N/A", "NA", "-", "--"]
    df_clean: pd.DataFrame = df.replace(invalid_values, np.nan)
    
    # Remover linhas com qualquer NaN
    df_clean = df_clean.dropna(how="any")
    rows_after: int = len(df_clean)
    
    if rows_before > 0:
        removed: int = rows_before - rows_after
        percentage: float = (removed / rows_before) * 100
        print(f"\n{name}:")
        print(f"  Linhas antes: {rows_before}")
        print(f"  Linhas após limpeza: {rows_after}")
        print(f"  Linhas removidas: {removed}")
        print(f"  Percentual removido: {percentage:.2f}%")
    return df_clean
    

def show_descriptive_stats(datasets: Dict[str, pd.DataFrame]) -> None:
    """Mostra estatísticas descritivas de múltiplos datasets"""
    print("\n" + "="*60)
    print("ANÁLISE DESCRITIVA DOS DATASETS")
    print("="*60)
    for name, df in datasets.items():
        print(f"\n=== {name.upper()} ===")
        print(df.describe())

# ============================================================
# FUNÇÕES DE VISUALIZAÇÃO
# ============================================================

def plot_time_series(
    datasets: Dict[str, pd.DataFrame],
    titles: Optional[List[str]] = None,
    colors: Optional[List[str]] = None
) -> None:
    """Plota séries temporais de múltiplos datasets"""
    n_datasets: int = len(datasets)
    fig, axes = plt.subplots(n_datasets, 1, figsize=(12, 3.5*n_datasets))
    
    if n_datasets == 1:
        axes = [axes]
    
    default_colors: List[str] = ['blue', 'orange', 'green']
    
    for idx, (name, df) in enumerate(datasets.items()):
        color: str = colors[idx] if colors else default_colors[idx]
        title: str = titles[idx] if titles else f'Série Temporal - {name}'
        
        df.plot(ax=axes[idx], marker='o', linewidth=2, markersize=4, color=color)
        axes[idx].set_title(title, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Índice')
        axes[idx].set_ylabel('Valor')
        axes[idx].grid(True, alpha=0.3)
        axes[idx].legend(loc='best')
    
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_histograms(
    datasets: Dict[str, pd.DataFrame],
    titles: Optional[List[str]] = None,
    colors: Optional[List[str]] = None
) -> None:
    """Plota histogramas de múltiplos datasets"""
    n_datasets: int = len(datasets)
    fig, axes = plt.subplots(1, n_datasets, figsize=(5*n_datasets, 4))
    
    if n_datasets == 1:
        axes = [axes]
    
    default_colors: List[str] = ['blue', 'orange', 'green']
    
    for idx, (name, df) in enumerate(datasets.items()):
        color: str = colors[idx] if colors else default_colors[idx]
        title: str = titles[idx] if titles else f'Histograma - {name}'
        
        df.hist(ax=axes[idx], bins=15, edgecolor='black', alpha=0.7, color=color)
        axes[idx].set_title(title, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Valor')
        axes[idx].set_ylabel('Frequência')
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================

# Carregar dados
file_paths: Dict[str, str] = {
    "prod_agricola": "data/prod_agricola_2002_2024.csv",
    "prod_industrial": "data/prod_industrial_2002_2024.csv",
    "pib": "data/pib_2022_2024.csv"
}

datasets: Dict[str, pd.DataFrame] = load_data(file_paths)

# Mostrar estatísticas descritivas
show_descriptive_stats(datasets)

# Aplicar dropna e mostrar porcentagem removida
print("\n" + "="*60)
print("ANÁLISE DE VALORES FALTANTES (dropna)")
print("="*60)
datasets_clean: Dict[str, pd.DataFrame] = {}
for name, df in datasets.items():
    datasets_clean[name] = clean_data_with_dropna(df, name=name.upper())

# Converter últimas 3 colunas para numéricas
for name, df in datasets.items():
    print(f"{name}: {df.head(10)}")


    if name == "prod_industrial":
        n: int = 3
        df.to_csv("teste_ind.csv", index=False)
    else:
        n = 1
    datasets[name] = convert_last_n_columns_to_numeric(df, n)

print("Tipos de dado após conversão:")
for name, df in datasets.items():
    print(f"\n{name}:")
    print(df.dtypes)



# Plotar séries
print("\n" + "="*60)
print("PLOTANDO SÉRIES TEMPORAIS")
print("="*60)
plot_time_series(
    datasets,
    titles=[
        'Série Temporal - Produção Agrícola (2002-2024)',
        'Série Temporal - Produção Industrial (2002-2024)',
        'Série Temporal - PIB (2022-2024)'
    ],
    colors=['blue', 'orange', 'green']
)

# Plotar histogramas
print("\n" + "="*60)
print("PLOTANDO HISTOGRAMAS")
print("="*60)
plot_histograms(
    datasets,
    titles=[
        'Histograma - Produção Agrícola',
        'Histograma - Produção Industrial',
        'Histograma - PIB'
    ],
    colors=['blue', 'orange', 'green']
)


ANÁLISE DESCRITIVA DOS DATASETS

=== PROD_AGRICOLA ===
               Ano         Valor
count    69.000000  6.900000e+01
mean   2013.000000  1.309437e+08
std       6.681845  1.503467e+08
min    2002.000000  4.699400e+07
25%    2007.000000  5.905960e+07
50%    2013.000000  7.327434e+07
75%    2019.000000  9.165108e+07
max    2024.000000  7.163265e+08

=== PROD_INDUSTRIAL ===
                 Mês  Região  \
count           1656    1656   
unique           276       1   
top     janeiro 2002  Brasil   
freq               6    1656   

                                                Variável Indústria geral  \
count                                               1656            1656   
unique                                                 9             792   
top     PIMPF - Número-índice (2022=100) (Número-índice)             ...   
freq                                                 276              47   

       Indústrias extrativas Indústrias de transformação  
count                

TypeError: Invalid value '0        84.35985
1        90.77187
2         0.00000
3             NaN
4             NaN
          ...    
1651    103.66426
1652      0.00000
1653      1.40000
1654      3.10000
1655      3.10000
Name: Indústria geral, Length: 1656, dtype: float64' for dtype 'str'